In [0]:
import requests

url = "https://api.coingecko.com/api/v3/exchanges"

response = requests.get(url, timeout=10)

print("Status:", response.status_code)
print("Response:", response.text[:2000])

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    BooleanType,
    TimestampType
)

exchange_bronze_schema = StructType([
    StructField("exchange_id", StringType(), False),
    StructField("name", StringType(), False),
    StructField("year_established", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("description", StringType(), True),
    StructField("url", StringType(), True),
    StructField("image", StringType(), True),
    StructField("has_trading_incentive", BooleanType(), True),
    StructField("trust_score", IntegerType(), True),
    StructField("trust_score_rank", IntegerType(), True),
    StructField("trade_volume_24h_btc", DoubleType(), True),
    StructField("last_updated", TimestampType(), True),
    StructField("source_system", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False)
])

print("Exchange Bronze schema created successfully")

In [0]:
exchanges_data = response.json()

print("Records received:", len(exchanges_data))

In [0]:
from datetime import datetime, timezone

exchange_rows = []

for exchange in exchanges_data:

    last_updated = None

    ingestion_timestamp = datetime.now(timezone.utc)

    exchange_rows.append((
        exchange.get("id"),
        exchange.get("name"),
        exchange.get("year_established"),
        exchange.get("country"),
        exchange.get("description"),
        exchange.get("url"),
        exchange.get("image"),
        exchange.get("has_trading_incentive"),
        exchange.get("trust_score"),
        exchange.get("trust_score_rank"),
        exchange.get("trade_volume_24h_btc"),
        last_updated,
        "CoinGecko",
        ingestion_timestamp
    ))

print("Rows prepared:", len(exchange_rows))
print("Values per row:", len(exchange_rows[0]))

In [0]:
exchange_df = spark.createDataFrame(
    exchange_rows,
    schema=exchange_bronze_schema
)

print("Exchange DataFrame created successfully")

In [0]:
exchange_df.printSchema()

In [0]:
exchange_df.show(truncate=False)

In [0]:
target_table = "bronze_exchange_master"

exchange_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print(f"Data written successfully to {target_table}")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM bronze_exchange_master
""").show()

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT exchange_id) AS unique_exchange_ids,
    COUNT(*) - COUNT(DISTINCT exchange_id) AS duplicate_records,

    COUNT(*) FILTER (WHERE exchange_id IS NULL) AS null_exchange_ids,
    COUNT(*) FILTER (WHERE name IS NULL) AS null_names,

    COUNT(*) FILTER (WHERE trust_score IS NULL) AS null_trust_scores,
    COUNT(*) FILTER (WHERE trust_score < 0 OR trust_score > 10) AS invalid_trust_scores,

    COUNT(*) FILTER (WHERE trust_score_rank IS NULL) AS null_trust_ranks,

    COUNT(*) FILTER (WHERE trade_volume_24h_btc < 0) AS invalid_trade_volume,

    COUNT(*) FILTER (WHERE ingestion_timestamp IS NULL) AS null_ingestion_timestamp

FROM bronze_exchange_master;